## Data Pipelines - Prefect - Example and Workshop
### Big Data Tools 
#### M.Sc. in Applied Analytics (coterminal course)
Fac. de Ingeniería -  Universidad de la Sabana<br>
Prof.: Hugo Franco, Ph.D.

In [ ]:
import os
os.environ['OMP_NUM_THREADS']='1' #prevent problems with 

from prefect import flow, task, get_run_logger
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from kaggle import KaggleApi
import logging

LOCAL_KAGGLE_DIR = './auth'
DOWNLOAD_DIRECTORY = './data'
TARGET_DATASET='sushilyeotiwad/wheat-seed-dataset'
# 2. Create the download directory if it doesn't exist
if not os.path.exists(DOWNLOAD_DIRECTORY):
    print(f"Creating directory: {DOWNLOAD_DIRECTORY}")
    os.makedirs(DOWNLOAD_DIRECTORY)

# 3. Authenticate in Kaggle to access the working dataset
@task(retries=3, retry_delay_seconds=10, name="Authenticate")
def api_authenticate():
    api=KaggleApi()
    api.authenticate()
    return api

@flow
def extract_data():
    download_path=DOWNLOAD_DIRECTORY
    dataset_id=TARGET_DATASET
    logging.log(f"Downloading dataset '{dataset_id}' to '{download_path}'...")
    kaggle_api=api_authenticate()
# 4. Download the dataset files and unzip them
    kaggle_api.dataset_download_files(dataset_id, path=download_path, unzip=True)
    # Load the dataframe.

os.chdir(DOWNLOAD_DIRECTORY)
df_seeds = pd.read_csv('seeds_dataset.csv')
# Basic data cleansing: Exclude the 'Class' column as it's the target variable and we're looking for independent variables for clustering.
df_clustering = df_seeds.drop('Class_(1, 2, 3)', axis=1)


### Basic (Unsupervised) Machine Learning analytics method
 Determine, by clustering, a sound number of species within the wheat seed dataset

 Firstly, identify the less correlated pair of features and use them as the representation for the instances in the dataset

In [ ]:
# Calculate the correlation matrix.
correlation_matrix = df_clustering.corr()

# Find the pair of variables with the lowest absolute correlation.
# We take the upper triangle of the correlation matrix and flatten it, and ignore the diagonal (which are all 1).
# We then find the minimum value and the corresponding row and column.
lower_triangle = correlation_matrix.mask(np.triu(np.ones(correlation_matrix.shape)).astype(bool))
lowest_corr = lower_triangle.stack().abs().min()
lowest_corr_pair = lower_triangle.stack().abs().idxmin()

# Print the lowest correlation and the corresponding pair of variables.
print(f"The lowest absolute correlation is {lowest_corr:.4f}, between {lowest_corr_pair}")

# Plot a heatmap of the correlation matrix for visualization.
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Seed Features')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('correlation_heatmap.png')
plt.show()

Now, use the `KMeans` clustering method to perform an unsupervised approach to the identification of the number of species in the dataset, according to the observations 

In [ ]:
# Select the two variables for clustering.
X = df_clustering[['Asymmetry_coefficient', 'Length_of_kernel_groove']]

# Determine the optimal number of clusters using the Elbow Method.
inertia = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)

# Plot the Elbow Method results.
plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), inertia, marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(range(1, 11))
plt.grid(True)
plt.savefig('elbow_method.png')
plt.show()

# Based on the Elbow plot, choose the optimal number of clusters.
# For this dataset, the 'Class' column suggests ?? classes, and the elbow method plot shows a good elbow at k=??.
optimal_k = 3

# Perform K-means clustering with the optimal number of clusters.
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_clustering['cluster'] = kmeans_model.fit_predict(X)

# Visualize the clustering results.
plt.figure(figsize=(10, 8))
scatter = plt.scatter(df_clustering['Asymmetry_coefficient'], df_clustering['Length_of_kernel_groove'], c=df_clustering['cluster'], cmap='viridis')
plt.title('K-means Clustering of Seeds (K=3)')
plt.xlabel('Asymmetry_coefficient')
plt.ylabel('Length_of_kernel_groove')
plt.legend(*scatter.legend_elements(), title="Clusters")
plt.grid(True)
plt.savefig('kmeans_clustering_plot.png')
plt.show()



__Challenges (Workshop 3, part 2):__
1. Organize the code of the example and complete the data pipeline: 
    - Use `try-except-finally` blocks as required 
    - Every individual process must be wrapped as a `task` using the corresponding decorator and its parameters when necessary. 
    - Create a `transform_data` task focused on data cleansing
    - Create a `load_data` task to create a table in a Dockerized PostgreSQL database and populate with with the clustering-oriented Dataframe
    - Invoke the tasks in the proper order in the `flow`
    - Use the `timing_decorator` to report the duration of each task 
1. Using the function get_directory_size, create a decorator to get and report the size of the downloaded dataset (size in bytes of the download folder)
1. Report the size in the previous question in a human readable unit
1. Use the attribute kmeans.cluster_centers_ and scatter plot to add the centroids of the best KMeans model (best K parameter) to the scatter plot
    - `centroids = kmeans.cluster_centers_`


In [ ]:
def get_directory_size(path='.'):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            # Create the full file path by joining the directory path and file name
            fp = os.path.join(dirpath, f)
            # Skip symbolic links to avoid counting them
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size    